# Opportunity · Cost & Performance Leakage

Two slow leaks: **fees** you pay every year regardless of performance, and **active funds**
that charge for a return an index already delivers. Small percentages compound into real
money over a holding period.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger, mapping, opportunities as O

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
entries, _, _ = ledger.load()
account_meta = mapping.load_account_meta()

TODAY = pd.Timestamp.today().normalize()
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} · {positions['account_name'].nunique()} accounts · as of {TODAY:%Y-%m-%d}")

## Fund expense drag

Annual dollars paid in fund fees (value × expense ratio). Individual stocks have none — so
this is really about the fund sleeve. Ratios come from `symbol_map.yaml`; the proprietary
401(k) pools are **estimates** — confirm them against your plan documents.

In [ ]:
ed = O.expense_drag(positions).reset_index()
annual = ed["annual_cost"].sum()
fig = px.bar(ed, x="annual_cost", y="symbol", orientation="h", color="expense_ratio",
             color_continuous_scale="Oranges", text="description",
             hover_data={"market_value": ":$,.0f", "expense_ratio": ":.2%"},
             title=f"Annual fund fees ≈ ${annual:,.0f}/yr")
fig.update_layout(height=360, xaxis_tickprefix="$", yaxis_title="",
                  coloraxis_colorbar_tickformat=".2%")
fig.update_traces(textposition="outside")
fig.show()
yrs, grow = 10, 0.06
compounded = annual * sum((1 + grow) ** y for y in range(yrs))
print(f"At ~{annual:,.0f}/yr, fees compound to ≈${compounded:,.0f} over {yrs} years "
      f"(assuming the fee base grows ~{grow:.0%}/yr).")

## Active vs. index — are you paying for alpha you're getting?

Each fund/holding's annualized return vs the index benchmark for its asset class. **Below
the diagonal = lagging** its benchmark — a swap candidate (especially if it also charges a
fee). Proprietary funds are shown via their tracking proxy.

In [ ]:
bg = O.benchmark_gap(positions, history)
lim = [min(bg[["holding_cagr", "benchmark_cagr"]].min()) - 0.02,
       max(bg[["holding_cagr", "benchmark_cagr"]].max()) + 0.02]
fig = px.scatter(bg, x="benchmark_cagr", y="holding_cagr", text="symbol", color="excess",
                 color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
                 hover_data={"benchmark": True, "info_ratio": ":.2f", "tracking_error": ":.1%"},
                 title="Holding return vs its benchmark (above line = beating it)")
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(dash="dot", color="grey"), showlegend=False)
fig.update_traces(textposition="top center", selector=dict(mode="markers+text"))
fig.update_layout(height=520, xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                  xaxis_title="benchmark CAGR", yaxis_title="holding CAGR",
                  xaxis_range=lim, yaxis_range=lim)
fig.show()
lag = bg[bg["excess"] < 0]
if not lag.empty:
    print("Lagging their benchmark:")
    display(lag[["symbol", "asset_class", "benchmark", "holding_cagr", "benchmark_cagr", "excess"]]
            .style.format({"holding_cagr": "{:.1%}", "benchmark_cagr": "{:.1%}", "excess": "{:+.1%}"}))

## Trading fees & commissions paid

What you've actually paid in fees on transactions, by year — usually small in a no-commission
era, but worth a glance for option/ADR/wire fees.

In [ ]:
fees = A.fees_paid(transactions, freq="Y")
if fees.empty or fees.abs().sum() == 0:
    print("✓ No trading fees recorded in the ledger.")
else:
    fees.index = fees.index.astype(str)
    fig = px.bar(x=fees.index, y=fees.abs(), title=f"Fees paid by year (${fees.abs().sum():,.0f} total)",
                 color_discrete_sequence=[PALETTE[3]])
    fig.update_layout(height=340, yaxis_tickprefix="$", xaxis_title="", yaxis_title="fees")
    fig.show()

---
*These are **candidate generators**, not advice — every row is a starting point for your
own judgment (and, for anything tax-related, a CPA's). Prices are research-grade
(yfinance); cost basis and lots come from the curated ledger, so they're only as right as
its fixups.*
*Expense ratios for the proprietary pools are estimates in `symbol_map.yaml` — the dollar
drag scales directly with them. The active-vs-index gap uses proxy history for un-quoted
funds, so it reflects the proxy's path, not the exact share class.*